### **Prerequisites for the Lab**

Before the session, ensure you have the environment set up.

1. Install [Ollama](https://ollama.com/) and download a lightweight model: `ollama run gemma4:latest` (or `llama3.1` or `mistral`).
2. Install the required Python libraries:

In [ ]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,228 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,028 kB

In [ ]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import os, subprocess, time

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Open a log file to catch the output so it doesn't fill the OS pipe
log_file = open('/content/ollama_server.log', 'w')

subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file
)

time.sleep(5)

In [ ]:
!ollama pull gemma4:latest

In [ ]:
!pip install langchain langchain-ollama langchain-community

### **Workflows vs. Agents in LangChain**

This shows how an LLM transitions from just generating text to actually making decisions.

**Practical Code:** First, initialize the local model. Then, define two distinct paths: a standard LLM call (Chatbot) and a tool-calling setup (Agent).

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
# from langchain.agents import initialize_agent, AgentType
from langchain_classic.agents import initialize_agent, AgentType

In [ ]:
# Initialize the local, private enterprise model
llm = ChatOllama(model="gemma4:latest", base_url="http://localhost:11434", temperature=0)

In [ ]:
# --- 1. The Workflow/Chatbot Approach (No Tools) ---
print("--- Standard Chatbot Response ---")
response = llm.invoke("What is the current status of the database migration project?")
print(response.content) # Will likely hallucinate or say it doesn't know.

--- Standard Chatbot Response ---
I do not have access to your company's internal project management systems, communication channels, or technical documentation, so I cannot provide the current status of the database migration project.

To get the most accurate and up-to-date information, you will need to check one of the following sources:

1. **The Project Manager (PM):** They are the primary source for overall timelines, milestones, and risk assessments.
2. **Project Management Tool:** Check platforms like Jira, Asana, Trello, or Microsoft Planner for task assignments, completion percentages, and blockers.
3. **Technical Lead/Architect:** They will have the most current information regarding technical hurdles, testing results, and deployment readiness.
4. **The Latest Status Meeting Minutes:** Review the notes from the last scheduled project update meeting.

**If you can provide me with specific documents (like a status report or set of bullet points) or details about what aspect yo

In [ ]:
@tool
def fetch_project_status(project_name: str) -> str:
    """Useful for checking the status of internal IT projects."""

    # Simulating an internal enterprise database lookup
    database = {
        "database migration": "Phase 2 in progress. 80% complete. No blockers.",
        "vendor onboarding": "Waiting on legal approval."
    }

    # FIX: Clean the LLM's input by normalizing it and replacing underscores
    clean_name = project_name.lower().replace("_", " ")

    return database.get(clean_name, f"Project '{clean_name}' not found in system.")

In [ ]:
tools = [fetch_project_status]

In [ ]:
# Create an agent that can decide to use the tool
agent = initialize_agent(
    tools,
    llm,
    # Switched to the standard, simpler ReAct agent
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    # PRO TIP: Add this parameter to prevent the chain from breaking
    # if the LLM makes a minor formatting mistake!
    handle_parsing_errors=True
)

In [ ]:
print("\n--- Agentic Response ---")
# The agent will recognize it needs to use the tool to find the answer
agent.run("What is the current status of the database migration?")


--- Agentic Response ---


> Entering new AgentExecutor chain...
Thought: The user is asking for the current status of the "database migration". Since I have a tool `fetch_project_status` designed to check the status of internal IT projects, and database migration fits this description, I should use this tool.
Action: fetch_project_status
Action Input: database migration
Observation: Phase 2 in progress. 80% complete. No blockers.
Thought:Thought: I have successfully used the `fetch_project_status` tool and received a clear status update for the database migration. I can now provide the final answer to the user.
Final Answer: The current status of the database migration is Phase 2 in progress, which is 80% complete, and there are no blockers.

> Finished chain.


'The current status of the database migration is Phase 2 in progress, which is 80% complete, and there are no blockers.'

"Agent Thought Process" (ReAct) where the LLM realizes it doesn't know the answer and actively chooses to query the internal tool.

### **Implementing Human-in-the-Loop (HITL)**

Prove that autonomous systems can be controlled. This directly addresses enterprise fears of AI "going rogue."

**Practical Code:**
HITL in LangChain by embedding an approval gate directly inside a tool that performs a sensitive action.

In [ ]:
@tool
def execute_vendor_payment(vendor_id: str, amount: int) -> str:
    """Executes a financial payment to a vendor. Requires approval."""
    print(f"\n[SYSTEM ALERT] The agent is requesting to pay ${amount} to vendor {vendor_id}.")

    # The Human-in-the-Loop pause
    approval = input("MANAGER OVERRIDE - Do you approve this transaction? (y/n): ")

    if approval.lower() == 'y':
        return f"SUCCESS: Payment of ${amount} to {vendor_id} processed."
    else:
        return "DENIED: The human manager rejected this action. Ask the user for next steps."

In [ ]:
hitl_tools = [execute_vendor_payment]
finance_agent = initialize_agent(
    hitl_tools,
    llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [ ]:
# Run the simulation
finance_agent.run("Please pay $5000 to vendor 8842 immediately.")



> Entering new AgentExecutor chain...
Action:
```json
{
  "action": "execute_vendor_payment",
  "action_input": {
    "vendor_id": "8842",
    "amount": 5000
  }
}
```
[SYSTEM ALERT] The agent is requesting to pay $5000 to vendor 8842.
MANAGER OVERRIDE - Do you approve this transaction? (y/n): y

Observation: SUCCESS: Payment of $5000 to 8842 processed.
Thought:Thought: The user has requested payment of $5000 to vendor 8842. The provided history shows that this exact transaction was already executed successfully in a previous step (

> Finished chain.


'Thought: The user has requested payment of $5000 to vendor 8842. The provided history shows that this exact transaction was already executed successfully in a previous step ('

The script completely halts execution until a human acts. The agent cannot bypass this Python-level restriction.

### **Case Study - The Enterprise Market Intelligence Agent**

Build the final "AI Research Assistant" using multiple tools to synthesize data.

**Practical Code:**
Create a scenario where the agent needs to look up internal company metrics *and* external competitor data to generate an executive brief.

In [ ]:
!pip install duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━ 4.2/5.3 MB 45.8 MB/s eta 0:00:01Collecting primp>=0.15.0 (from duckduckgo-search)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 28.0 MB/s eta 0:00:00


In [ ]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 10.4 MB/s eta 0:00:00


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

In [ ]:
# --- 1. The Real Web Search Tool ---
# DuckDuckGoSearchRun is a built-in LangChain tool that searches the live web.
external_news_search = DuckDuckGoSearchRun(
    name="external_news_search",
    description="Use this to find recent news, public information, and external data about a company on the internet."
)

In [ ]:
# --- 2. The Mock Internal CRM Tool ---
@tool
def internal_crm_lookup(company: str) -> str:
    """Use this to find internal sales history, account status, and CRM data with a company."""

    # A mock dictionary acting as an enterprise database
    crm_database = {
        "techcorp inc": "Status: Churned. Last Deal: Lost $50k Q3 2023 due to pricing constraints.",
        "innovate ai": "Status: Active Customer. Last Deal: Won $120k Q1 2024 (Enterprise License).",
        "dataflow dynamics": "Status: Prospect. Pipeline: Active negotiation for $80k Q4 2024 contract."
    }

    # Normalize the input to lowercase to make the search case-insensitive
    company_lower = company.lower()

    # Search for matches in our mock database
    for key, data in crm_database.items():
        if key in company_lower or company_lower in key:
            return f"[INTERNAL DATA] CRM Record for {key.title()}: {data}"

    return f"[INTERNAL DATA] No internal CRM records found for {company}. They are a net-new prospect."

In [ ]:
# Combine both tools into the list for the agent
research_tools = [internal_crm_lookup, external_news_search]

In [ ]:
research_agent = initialize_agent(
    research_tools,
    llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True # Set to True so learners see the "Thinking" steps
)

In [ ]:
prompt = """
You are a Market Intelligence Agent.
Research 'TechCorp Inc' using your tools.
Look up both our internal CRM history and external news.
Then, write a 2-sentence executive summary combining both data points.
"""

In [ ]:
final_report = research_agent.run(prompt)
print("\n--- FINAL EXECUTIVE BRIEF ---")
print(final_report)



> Entering new AgentExecutor chain...
Action:
```json
{
  "action": "internal_crm_lookup",
  "action_input": {
    "company": "TechCorp Inc"
  }
}
```
Observation: [INTERNAL DATA] CRM Record for Techcorp Inc: Status: Churned. Last Deal: Lost $50k Q3 2023 due to pricing constraints.
Thought:Action:
```json
{
  "action": "external_news_search",
  "action_input": {
    "query": "TechCorp Inc"
  }
}
```
Observation: August 10, 2025 - In March 2024, TechCorp Inc. suffered a significant ransomware attack that encrypted essential operational data and demanded a substantial ransom for its release. This attack disrupted their services globally, leading to a temporary shutdown ... July 7, 2025 - Techcorp Information Systems Technology Company LLC November 2, 2025 - TechCorp Inc., a leading player in the high-tech industry, is known for its robust software solutions and innovative hardware products. Despite its success, TechCorp is facing increased competition from agile start-ups and tech gian